## Cloud-Based Intelligent Resource Allocation System for Campus Facilities (Phase 1)

**Domain:** Education  

**Framework:** LangChain  

**Model:** Anthropic Claude Sonnet 4.5 (`claude-sonnet-4.5`)

**Approach:** Round Robin (Next Speaker Selection)   

**Purpose:** Simulate Waterfall SDLC phases using Round Robin agent interaction with LangChain

## Environment Setup

Install required libraries, configure the Anthropic API key, and initialize the LangChain LLM settings

In [1]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "langchain", "langchain-anthropic", "langchain-core",
    "anthropic", "--quiet"
])
print("Libraries installed.")

Libraries installed.


In [2]:
import os, json, time, re
from datetime import datetime
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load API key from .env file
env_path = os.path.join(os.getcwd(), ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found. Add to .env: ANTHROPIC_API_KEY=sk-ant-...")

MODEL = "claude-sonnet-4-5"

llm = ChatAnthropic(
    model=MODEL,
    temperature=0.3,
    anthropic_api_key=ANTHROPIC_API_KEY,
    max_tokens=16000,
)

token_usage = {}
total_tokens = 0
start_time = time.time()

print(f"API Key loaded. Model: {MODEL}")
print(f"Framework: LangChain | Approach: Round Robin (Next Speaker Selection)")

API Key loaded. Model: claude-sonnet-4-5
Framework: LangChain | Approach: Round Robin (Next Speaker Selection)


## Project Configuration

Project details and realistic system boundaries are defined here. Appropriate scope is defined and shared with the engineering agents.

In [3]:
DOMAIN = """
PROJECT: Cloud-Based Intelligent Resource Allocation System for Campus Facilities
DOMAIN: Education
DESCRIPTION: Students and faculty book rooms, labs, and equipment online.
Administrators view real-time occupancy dashboards. AI/ML predicts demand and
auto-allocates resources. Integrates with Google Calendar, Microsoft Outlook,
and SSO (Single Sign-On). Sends booking/conflict notifications. Generates
utilization reports for facility management. Supports 5,000+ concurrent users
and maintains 99.9% uptime.
STAKEHOLDERS: Students, Faculty, Facility Admins, IT Department, University Management
FACILITIES: Classrooms, Computer Labs, Research Labs, Conference Rooms, Study Rooms
"""

PROJECT_SCOPE = """
Project scope and implementation boundaries:
- Total SLOC must be between 12,000 and 22,000
- This is a mid-size cloud-based campus facility management system
- Frontend (React/Next.js): 3,000-5,000 SLOC
- Backend API (Node.js/FastAPI): 4,000-7,000 SLOC
- AI/ML Module: 2,000-4,000 SLOC
- Database layer: 1,500-2,500 SLOC
- Integration layer (SSO, Calendar): 1,000-2,000 SLOC
CONSISTENCY CHECK before printing metrics:
  SLOC-based days  = Total SLOC / 50
  Hours-based days = Sum of all task hours / 8
  These two numbers must be within 15% of each other.
  If not, fix task hours first. Then print:
  CONSISTENCY CHECK: [SLOC days] = [Hours days] ✓
"""

print("Project configuration loaded.")

Project configuration loaded.


## Define All AI Agents

Define all AI agents for the Waterfall SDLC simulation using LangChain Round Robin (Next Speaker Selection).

- **Customer Proxy** (RoundRobinAgent) — presents the problem statement to the Project Manager
- **Project Manager** (RoundRobinAgent) — receives and coordinates each engineering agent in rotation
- **Requirements Engineer** (RoundRobinAgent) — generates use cases, functional and non-functional requirements
- **System Engineer** (RoundRobinAgent) — creates system architecture and design document
- **Software Developer** (RoundRobinAgent) — defines development tasks and SLOC estimates
- **Test Engineer** (RoundRobinAgent) — creates test cases and testing strategy
- **Documentation Engineer** (RoundRobinAgent) — creates end-user and technical documentation plan

In [4]:
class RoundRobinAgent:
    def __init__(self, name, system_prompt):
        self.name = name
        self.system_prompt = system_prompt
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "CONVERSATION HISTORY:\n{history}\n\nTASK:\n{input}")
        ])
        self.chain = self.prompt | llm | StrOutputParser()

    def respond(self, task, history=""):
        global total_tokens
        response = self.chain.invoke({"input": task, "history": history[-3000:]})
        approx = len(self.system_prompt.split()) + len(history[-3000:].split()) + len(response.split())
        token_usage[self.name] = token_usage.get(self.name, 0) + approx
        total_tokens += approx
        print(f"  [{self.name}] ~{approx:,} tokens")
        return response

# ── Agent 1: Customer Proxy ──────────────────────────────────────────
customer_proxy = RoundRobinAgent(
    name="Customer_Proxy",
    system_prompt=f"""You are the Customer Proxy AI Agent for the Waterfall SDLC.
Your role is to communicate the customer's requirements clearly to the Project Manager.
{DOMAIN}"""
)

# ── Agent 2: Project Manager ─────────────────────────────────────────
project_manager = RoundRobinAgent(
    name="Project_Manager",
    system_prompt=f"""You are the Project Manager AI Agent for the Waterfall SDLC.
You have received the customer request. Briefly acknowledge it, state the 5 Waterfall
phases that will be executed (Requirements, Design, Implementation, Testing, Documentation),
and confirm you are delegating to each specialist agent in order.
Keep your response concise — under 200 words.
{DOMAIN}"""
)

# ── Agent 3: Requirements Engineer ───────────────────────────────────
requirement_engineer = RoundRobinAgent(
    name="Requirement_Engineer",
    system_prompt=f"""You are the Requirements Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Create a COMPLETE use case list with unique IDs (UC-001, UC-002, ...) — at least 15 use cases
2. For each use case provide: Use Case ID, Name, Primary Actor, Preconditions,
   Postconditions, Main Flow, Alternate Flows
3. Create FUNCTIONAL requirements (FR-001, FR-002, ...) — at least 40 requirements
4. Create NON-FUNCTIONAL requirements (NFR-001, NFR-002, ...) — at least 15 requirements
5. All requirement IDs are sequential across the entire document
6. Include review and rework tasks (10% buffer)

STRICT CONSTRAINTS:
- Generate EXACTLY between 25 and 45 tasks. Do NOT exceed 45.
- Use ONLY these 5 SLOC categories — Frontend, Backend API, AI/ML, Database, Integration. No others. Do NOT add Testing SLOC, DevOps SLOC, Code Review SLOC.
- Frontend 3,000–5,000 | Backend 4,000–7,000 | AI/ML 2,000–4,000 | Database 1,500–2,500 | Integration 1,000–2,000.
- Do NOT use markdown bold (**text**) anywhere in the metrics block. Use plain text only.
- Number of Tasks MUST be between 6 and 10.
- Tasks must include: Requirements Gathering, Use Case Documentation,
  Functional Requirements Documentation, Non-Functional Requirements
  Documentation, Requirements Review, Requirements Rework.

Then calculate effort EXACTLY as follows:
- Step 1: Work = Total FR + NFR only (do NOT include use cases in the count)
- Step 2: Productivity rate = 5 requirements per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
– Step 6: Earliest Start = Day 1
– Step 7: Earliest Finish = Day 1 + Total Effort WITH Review - 1 (round up to whole number)

End your response with EXACTLY this block (real numbers only, no placeholders):
=== REQUIREMENTS PHASE METRICS ===
Total Use Cases:
Total Functional Requirements (FR):
Total Non-Functional Requirements (NFR):
Total Requirements (FR + NFR):
Productivity Rate: 5 requirements per day
Base Effort: [Total REQs] / 5 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day 1
Earliest Finish: Day [X]
Number of Tasks in Requirements Phase:
================================"""
)

# ── Agent 4: System Engineer ─────────────────────────────────────────
system_engineer = RoundRobinAgent(
    name="System_Engineer",
    system_prompt=f"""You are the System Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Create a HIGH-LEVEL SYSTEM ARCHITECTURE with layered design:
   (Client Layer, API/Orchestration Layer, AI/Intelligence Layer,
   Storage Layer, Cross-Cutting Services)
2. Provide a COMPONENT DIAGRAM (textual description)
3. Provide a SEQUENCE DIAGRAM for the Room Booking use case
4. Create an ENTITY-RELATIONSHIP DIAGRAM (textual) with all entities
5. Describe the TECHNOLOGY STACK recommendations
6. Design the AI/ML demand prediction module architecture
7. Design SSO and Calendar integration architecture
8. Reference specific FR and NFR IDs from the requirements document
9. Include review and rework tasks (10% buffer)

STRICT CONSTRAINTS:
- Number of Tasks MUST be between 6 and 10.
- Tasks must include: Architecture Design, Component Diagram,
  Sequence Diagram, ERD Design, Technology Stack Definition,
  AI/ML Module Design, SSO Integration Design, Design Review.
- Do NOT use markdown bold (**text**) anywhere in the metrics block. Use plain text only.

Then calculate effort EXACTLY:
- Step 1: Work = estimated total number of pages in design document
- Step 2: Productivity rate = 5 pages per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
– Step 6: Earliest Start = Requirements Earliest Finish + 1 (round up to whole number)
– Step 7: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)

End with EXACTLY:
=== DESIGN PHASE METRICS ===
Total Design Document Pages:
Architecture Diagrams Count:
Database Schema Pages:
API Documentation Pages:
UI Wireframe Pages:
Other Design Pages:
Productivity Rate: 5 pages per day
Base Effort: [Total Pages] / 5 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Design Phase:
============================"""
)

# ── Agent 5: Software Developer ──────────────────────────────────────
software_developer = RoundRobinAgent(
    name="Software_Developer",
    system_prompt=f"""You are the Software Developer AI Agent for the Waterfall SDLC.
{DOMAIN}
{PROJECT_SCOPE}

Your responsibilities:
1. List ALL development tasks with IDs (T-001, T-002, ...)
2. For each task: ID, Module, Description, SLOC estimate, Hours estimate, Linked FR IDs
3. Organize tasks by module:
   - User Auth & SSO Module
   - Facility Booking Module
   - Real-time Occupancy Dashboard
   - AI Demand Prediction Module
   - Notification & Alerts Module
   - Reporting & Analytics Module
   - Admin Management Module
   - API Gateway & Integration
4. Minimum 25 tasks, maximum 45 tasks
5. Include code review tasks (10% buffer)

STRICT CONSTRAINTS:
- Generate EXACTLY between 25 and 45 tasks. Do NOT exceed 45.
- Use ONLY these 5 SLOC categories — Frontend, Backend API, AI/ML, Database, Integration. No others. Do NOT add Testing SLOC, DevOps SLOC, Code Review SLOC.
- Frontend 3,000–5,000 | Backend 4,000–7,000 | AI/ML 2,000–4,000 | Database 1,500–2,500 | Integration 1,000–2,000.
- Do NOT use markdown bold (**text**) anywhere in the metrics block. Use plain text only.

Then calculate effort EXACTLY:
- Step 1: Work = total SLOC (MUST be between 12,000 and 22,000)
- Step 2: Productivity rate = 50 SLOC per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Sum all task hours once by adding each task's hours estimate.
          Do NOT repeat any numbers. Print only the final total sum.
          Then check: SLOC/50 × 8 ≈ Total Task Hours. Print result.
– Step 7: Earliest Start = Design Earliest Finish + 1 (round up to whole number)
– Step 8: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)


End with EXACTLY:
=== IMPLEMENTATION PHASE METRICS ===
Total Development Tasks:
Frontend SLOC:
Backend SLOC:
Database Scripts SLOC:
AI/ML Module SLOC:
Integration/API SLOC:
Total SLOC (MUST be 12,000–22,000):
Productivity Rate: 50 SLOC per day
Base Effort: [Total SLOC] / 50 = [X] days
Code Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Total Task Hours (sum of all tasks):
CONSISTENCY CHECK: [SLOC/50 × 8] = [X] hrs ≈ Total Task Hours [PASS/FAIL]
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Implementation Phase:
=================================="""
)

# ── Agent 6: Test Engineer ────────────────────────────────────────────
test_engineer = RoundRobinAgent(
    name="Test_Engineer",
    system_prompt=f"""You are the Test Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Define the overall TEST STRATEGY (levels: unit, integration, system, AI evaluation)
2. Create FUNCTIONAL TEST CASES (TC-001 through TC-XX) with:
   Related UC/FR IDs, Preconditions, Steps, Expected Results
3. Create NON-FUNCTIONAL TEST CASES (NTC-001 through NTC-XX) covering:
   Performance (5,000 concurrent users), AI Accuracy (>85% target),
   Availability (99.9% uptime), Security/RBAC, Usability
4. Provide a TESTING SCHEDULE broken into weeks
5. Include test review tasks (10% buffer)

STRICT CONSTRAINTS:
- NON-FUNCTIONAL TEST CASES must include ALL 5 categories separately:
  Performance (5,000 concurrent users)
  AI Accuracy (>85% target)
  Security/RBAC
  Availability (99.9% uptime)
  Usability
- List each category with its individual count in the metrics block.
- All 5 NTC subcategory counts MUST add up exactly to Total NTC number.
- Do NOT include UAT under Non-Functional — UAT belongs under Functional test cases.
- Number of Tasks MUST be between 8 and 15.
- Do NOT equate tasks with test cases.
- Tasks are activities such as: Test Planning, Test Case Design,
  Unit Test Execution, Integration Test Execution, System Test
  Execution, Performance Test Execution, UAT Execution,
  Test Reporting, Test Review.
- Do NOT use markdown bold (**text**) anywhere in the metrics block. Use plain text only.

Then calculate effort EXACTLY:
- Step 1: Work = total number of test cases
- Step 2: Productivity rate = 2 test cases per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
– Step 6: Earliest Start = Implementation Earliest Finish + 1 (round up to whole number)
– Step 7: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)

End with EXACTLY:
=== TESTING PHASE METRICS ===
Total Test Cases:
Functional Test Cases (TC):
Non-Functional Test Cases (NTC):
Unit Test Cases:
Integration Test Cases:
System/E2E Test Cases:
Performance Test Cases:
AI Accuracy Test Cases:
UAT Test Cases:
Productivity Rate: 2 test cases per day
Base Effort: [Total TCs] / 2 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Testing Phase:
============================"""
)

# ── Agent 7: Documentation Engineer ──────────────────────────────────
documentation_engineer = RoundRobinAgent(
    name="Documentation_Engineer",
    system_prompt=f"""You are the Documentation Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Create an END-USER DOCUMENTATION OUTLINE:
   - Student User Guide (booking, equipment, notifications)
   - Faculty User Guide (booking, recurring reservations)
   - Administrator Guide (dashboard, reports, AI settings)
   - AI System Explainer (how demand prediction works)
   - Troubleshooting & FAQ
2. Create a DEVELOPER DOCUMENTATION OUTLINE:
   - System Architecture Overview
   - RESTful & GraphQL API Reference
   - Database Schema & ERD
   - AI/ML Model Documentation
   - Deployment & CI/CD Guide
   - Security & Compliance Guide
3. Create TRAINING MATERIALS outline:
   - Student Onboarding (pages estimate)
   - Admin Training Manual (pages estimate)
4. Estimate pages per section
5. Include documentation review tasks (10% buffer)

STRICT CONSTRAINTS:
- End-User Documentation: MAXIMUM 50 pages
- Technical/Developer Documentation: MAXIMUM 60 pages
- Training Materials: MAXIMUM 35 pages
- Total Documentation Pages MUST be between 90 and 145 pages.
- Keep decimals through ALL calculation steps.
  Do NOT round Base Effort or Review Effort separately.
  Apply ceil() ONCE to the final Total Effort WITH Review only.
- Do NOT use markdown bold (**text**) anywhere in the metrics block. Use plain text only.

Then calculate effort EXACTLY:
- Step 1: Work = total documentation pages
- Step 2: Productivity rate = 3 pages per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
– Step 6: Earliest Start = Testing Earliest Finish + 1 (round up to whole number)
– Step 7: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)

End with EXACTLY:
=== DOCUMENTATION PHASE METRICS ===
End-User Documentation Pages:
Technical/Developer Documentation Pages:
Training Material Pages:
Total Documentation Pages:
Productivity Rate: 3 pages per day
Base Effort: [Total Pages] / 3 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Documentation Phase:
=================================="""
)

print("All 7 agents created successfully (LangChain Round Robin).")
print("  - Customer_Proxy         (RoundRobinAgent)")
print("  - Project_Manager        (RoundRobinAgent)")
print("  - Requirement_Engineer   (RoundRobinAgent)")
print("  - System_Engineer        (RoundRobinAgent)")
print("  - Software_Developer     (RoundRobinAgent)")
print("  - Test_Engineer          (RoundRobinAgent)")
print("  - Documentation_Engineer (RoundRobinAgent)")

All 7 agents created successfully (LangChain Round Robin).
  - Customer_Proxy         (RoundRobinAgent)
  - Project_Manager        (RoundRobinAgent)
  - Requirement_Engineer   (RoundRobinAgent)
  - System_Engineer        (RoundRobinAgent)
  - Software_Developer     (RoundRobinAgent)
  - Test_Engineer          (RoundRobinAgent)
  - Documentation_Engineer (RoundRobinAgent)


## Customer Problem Statement

The customer sends the initial requirements to the Project Manager

In [5]:
customer_message = (
    "I want to create a Cloud-Based Intelligent Resource Allocation System "
    "for Campus Facilities at our university. The system must:\n"
    "1. Allow students and faculty to book rooms, labs, and equipment online\n"
    "2. Allow administrators to view real-time occupancy dashboards\n"
    "3. Use AI/ML to predict demand and auto-allocate resources\n"
    "4. Integrate with Google Calendar, Microsoft Outlook, and SSO\n"
    "5. Send notifications for booking confirmations and conflicts\n"
    "6. Generate utilization reports for facility management\n"
    "7. Support 5,000+ concurrent users and maintain 99.9% uptime\n"
    "Please generate the complete Waterfall project plan."
)

print("Customer message defined.")
print(f"Message length: {len(customer_message)} characters")

Customer message defined.
Message length: 615 characters


## Run Experiment

Each agent is selected as the next speaker by rotating through the agent list.

In [6]:
print("Starting LangChain Round Robin experiment...")
print("=" * 60)

history = ""
all_responses = {}

pm_to_requirement_engineer = (
    f"I have received the following message from a customer:\n"
    f"\"{customer_message}\"\n\n"
    f"Generate the complete requirements document. Give every use case a unique name "
    f"and number (UC-001, UC-002...) and every requirement a unique sequential number "
    f"(FR-001, FR-002...). Include at least 15 use cases, 40 functional requirements, "
    f"and 15 non-functional requirements. Include reviews and rework tasks (10% buffer). "
    f"Calculate effort using Work / Productivity Rate (5 reqs/day). "
    f"YOU MUST print the complete REQUIREMENTS PHASE METRICS block at the end."
)

pm_to_system_engineer = (
    f"The Requirements Engineer has completed the requirements document. "
    f"It is in the conversation history above.\n\n"
    f"Create the complete system design document. Include architecture (5 layers), "
    f"component diagram, sequence diagram, ERD, technology stack, AI/ML module design, "
    f"and SSO/Calendar integration. Include reviews and rework tasks (10% buffer). "
    f"Calculate effort using Work / Productivity Rate (5 pages/day). "
    f"IMPORTANT: Your Design phase Earliest Start = Requirements Earliest Finish Day + 1. "
    f"YOU MUST print the complete DESIGN PHASE METRICS block at the end."
)

pm_to_software_engineer = (
    f"The Requirements and System Engineers have completed their work. "
    f"Their documents with metrics are in the conversation history above.\n\n"
    f"Create the complete implementation plan. List every task with ID (T-001, T-002...), "
    f"module, description, SLOC estimate, hours estimate, and linked FR IDs. "
    f"STRICT: Total SLOC must be between 12,000 and 22,000. "
    f"Include code review tasks (10% buffer). "
    f"Calculate effort using Work / Productivity Rate (50 SLOC/day). "
    f"Run consistency check: SLOC-based days vs hours-based days (must be within 15%). "
    f"IMPORTANT: Your Implementation Earliest Start = Design Earliest Finish Day + 1. "
    f"YOU MUST print the complete IMPLEMENTATION PHASE METRICS block at the end."
)

pm_to_test_engineer = (
    f"The software development plan is complete. "
    f"All documents with metrics are in the conversation history above.\n\n"
    f"Create the complete test plan with functional test cases (TC-001...) and "
    f"non-functional test cases (NTC-001...). Include unit, integration, system, "
    f"performance (5,000 users), AI accuracy (>85%), security/RBAC, and UAT test cases. "
    f"Include test review tasks (10% buffer). "
    f"Calculate effort using Work / Productivity Rate (2 test cases/day). "
    f"IMPORTANT: Your Testing Earliest Start = Implementation Earliest Finish Day + 1. "
    f"YOU MUST print the complete TESTING PHASE METRICS block at the end."
)

pm_to_documentation_engineer = (
    f"Testing is complete. All previous phase metrics are in the conversation history above.\n\n"
    f"Create the complete documentation plan with page estimates for every section. "
    f"Include end-user docs (Student Guide, Faculty Guide, Admin Guide, AI Explainer, FAQ), "
    f"developer docs (Architecture, API Reference, DB Schema, AI/ML Docs, CI/CD, Security), "
    f"and training materials. Include documentation review tasks (10% buffer). "
    f"Calculate effort using Work / Productivity Rate (3 pages/day). "
    f"IMPORTANT: Your Documentation Earliest Start = Testing Earliest Finish Day + 1. "
    f"YOU MUST print the complete DOCUMENTATION PHASE METRICS block at the end."
)

# ── Round Robin sequence — next speaker selected by rotating through list ──
agents_in_order = [
    (customer_proxy,         "Customer_Proxy"),
    (project_manager,        "Project_Manager"),
    (requirement_engineer,   "Requirement_Engineer"),
    (system_engineer,        "System_Engineer"),
    (software_developer,     "Software_Developer"),
    (test_engineer,          "Test_Engineer"),
    (documentation_engineer, "Documentation_Engineer"),
]

# Define task for each agent 
def get_task(agent_name, customer_msg, history):
    if agent_name == "Customer_Proxy":
        return customer_msg
    elif agent_name == "Project_Manager":
        return (
            f"I have received the following message from the customer:\n"
            f"\"{customer_msg}\"\n\n"
            f"Acknowledge the request, list the 5 Waterfall phases, and confirm delegation."
        )
    elif agent_name == "Requirement_Engineer":
        return pm_to_requirement_engineer
    elif agent_name == "System_Engineer":
        return pm_to_system_engineer
    elif agent_name == "Software_Developer":
        return pm_to_software_engineer
    elif agent_name == "Test_Engineer":
        return pm_to_test_engineer
    elif agent_name == "Documentation_Engineer":
        return pm_to_documentation_engineer

# ── Execute Round Robin — rotating next speaker selection ──────────────
for i, (agent, agent_name) in enumerate(agents_in_order):
    print(f"\nRound {i+1}/7 → Next Speaker Selected: {agent_name}")
    print("-" * 50)

    task = get_task(agent_name, customer_message, history)
    response = agent.respond(task, history)

    # Force metrics block on second attempt if missing
    if agent_name not in ["Customer_Proxy", "Project_Manager"]:
        if "PHASE METRICS" not in response:
            print(f"  [{agent_name}] Metrics missing — requesting metrics block...")
            metrics_response = agent.respond(
                f"Output ONLY your phase metrics block. Start with === and end with ===. Use real numbers from your previous response. Nothing else before or after the metrics block.",
                history + f"\n{agent_name}:\n{response[-2000:]}\n"
            )
            response = response + "\n\n" + metrics_response

    history += f"\n{agent_name}:\n{response}\n"
    all_responses[agent_name] = response
    print(f"{agent_name} responded ({len(response):,} chars)")

total_elapsed = time.time() - start_time
total_tokens_final = sum(token_usage.values())

print(f"\n{'=' * 60}")
print(f"Round Robin pipeline completed in {total_elapsed:.1f}s")
print(f"{'=' * 60}")

Starting LangChain Round Robin experiment...

Round 1/7 → Next Speaker Selected: Customer_Proxy
--------------------------------------------------
  [Customer_Proxy] ~7,039 tokens
Customer_Proxy responded (55,370 chars)

Round 2/7 → Next Speaker Selected: Project_Manager
--------------------------------------------------
  [Project_Manager] ~791 tokens
Project_Manager responded (1,617 chars)

Round 3/7 → Next Speaker Selected: Requirement_Engineer
--------------------------------------------------
  [Requirement_Engineer] ~5,812 tokens
Requirement_Engineer responded (37,304 chars)

Round 4/7 → Next Speaker Selected: System_Engineer
--------------------------------------------------
  [System_Engineer] ~6,696 tokens
  [System_Engineer] Metrics missing — requesting metrics block...
  [System_Engineer] ~863 tokens
System_Engineer responded (67,344 chars)

Round 5/7 → Next Speaker Selected: Software_Developer
--------------------------------------------------
  [Software_Developer] ~3,397 

## Save Results

Save each agent's output as a markdown file and count tokens per agent.

In [7]:
import os, json
from datetime import datetime

output_dir = "Phase1_Experiment2_Results_LangChain"
os.makedirs(output_dir, exist_ok=True)

# Save individual agent outputs as .md files
for agent_name, response in all_responses.items():
    filepath = os.path.join(output_dir, f"{agent_name}_output.md")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# {agent_name.replace('_', ' ')} Output\n\n")
        f.write(f"**Model:** {MODEL} | **Tokens:** ~{token_usage.get(agent_name, 0):,}\n\n")
        f.write(response)
    print(f"Saved: {filepath}")

# Save combined report
section_names = {
    "Customer_Proxy":         "SECTION 0 ║ CUSTOMER PROXY",
    "Project_Manager":        "SECTION 1 ║ PROJECT MANAGER",
    "Requirement_Engineer":   "SECTION 2 ║ REQUIREMENTS ENGINEER",
    "System_Engineer":        "SECTION 3 ║ SYSTEM ENGINEER",
    "Software_Developer":     "SECTION 4 ║ SOFTWARE DEVELOPER",
    "Test_Engineer":          "SECTION 5 ║ TEST ENGINEER",
    "Documentation_Engineer": "SECTION 6 ║ DOCUMENTATION ENGINEER",
}

combined_path = os.path.join(output_dir, "Phase1_Waterfall_Complete_Report.md")
with open(combined_path, "w", encoding="utf-8") as f:
    f.write("# Phase 1 ║ Waterfall Project Plan\n")
    f.write("# Cloud-Based Intelligent Resource Allocation System for Campus Facilities\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"**LLM Model:** {MODEL}\n")
    f.write(f"**Framework:** LangChain — Round Robin (Next Speaker Selection)\n\n---\n\n")
    f.write("## SECTION 0 ║ CUSTOMER PROBLEM STATEMENT\n\n")
    f.write(customer_message.strip() + "\n\n---\n\n")
    for agent_name, response in all_responses.items():
        if agent_name in section_names:
            f.write(f"## {section_names[agent_name]}\n")
            f.write(f"**Model:** {MODEL} | **Tokens:** ~{token_usage.get(agent_name, 0):,}\n\n")
            f.write(response)
            f.write("\n\n---\n\n")

print(f"\nSaved combined report: {combined_path}")

# Save metadata JSON
metadata = {
    "experiment": {
        "title": "Phase 1 ║ Waterfall Project Planning",
        "project": "Cloud-Based Intelligent Resource Allocation System for Campus Facilities",
        "model": MODEL,
        "framework": "LangChain — Round Robin",
        "interaction_type": "Round Robin (Next Speaker Selection)",
        "temperature": 0.3,
        "date": datetime.now().isoformat(),
        "total_tokens": total_tokens_final,
        "total_time_seconds": round(total_elapsed, 2),
    },
    "agents": [
        {"name": k, "tokens": token_usage.get(k, 0), "model": MODEL}
        for k in all_responses.keys()
    ],
    "sequence": {
        "type": "round_robin",
        "total_agents": len(all_responses),
        "interaction": "Next speaker selected by rotating through agent list",
    },
}

metadata_path = os.path.join(output_dir, "experiment_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata: {metadata_path}")

Saved: Phase1_Experiment2_Results_LangChain\Customer_Proxy_output.md
Saved: Phase1_Experiment2_Results_LangChain\Project_Manager_output.md
Saved: Phase1_Experiment2_Results_LangChain\Requirement_Engineer_output.md
Saved: Phase1_Experiment2_Results_LangChain\System_Engineer_output.md
Saved: Phase1_Experiment2_Results_LangChain\Software_Developer_output.md
Saved: Phase1_Experiment2_Results_LangChain\Test_Engineer_output.md
Saved: Phase1_Experiment2_Results_LangChain\Documentation_Engineer_output.md

Saved combined report: Phase1_Experiment2_Results_LangChain\Phase1_Waterfall_Complete_Report.md
Saved metadata: Phase1_Experiment2_Results_LangChain\experiment_metadata.json


## Experiment Summary

Gives the final experiment results and per-agent token breakdown.

In [8]:
print("=" * 60)
print("  EXPERIMENT 2 COMPLETE")
print("=" * 60)
print(f"  Model:        {MODEL}")
print(f"  Framework:    LangChain")
print(f"  Interaction:  Round Robin (Next Speaker Selection)")
print(f"  Total Agents: {len(all_responses)}")
print(f"  Total Steps:  {len(all_responses)} (one per Waterfall phase)")
print(f"  Total Tokens: ~{total_tokens_final:,}")
print(f"  Total Time:   {total_elapsed:.1f}s")
print()
print("  Agent Breakdown:")
for agent_name, response in all_responses.items():
    tokens = token_usage.get(agent_name, 0)
    print(f"    {agent_name:30s} | Tokens: ~{tokens:6,} | Response: {len(response):6,} chars")
print()
print(f"  Output Directory: {output_dir}/")
print("=" * 60)

import re
print()
print("=" * 60)
print("  PHASE METRICS SUMMARY")
print("=" * 60)
metrics_patterns = {
    "Requirement_Engineer":   r"=== REQUIREMENTS PHASE METRICS ===(.*?)================================",
    "System_Engineer":        r"=== DESIGN PHASE METRICS ===(.*?)============================",
    "Software_Developer":     r"=== IMPLEMENTATION PHASE METRICS ===(.*?)==================================",
    "Test_Engineer":          r"=== TESTING PHASE METRICS ===(.*?)============================",
    "Documentation_Engineer": r"=== DOCUMENTATION PHASE METRICS ===(.*?)==================================",
}
for agent_name, pattern in metrics_patterns.items():
    if agent_name in all_responses:
        match = re.search(pattern, all_responses[agent_name], re.DOTALL)
        if not match:
            cleaned = all_responses[agent_name].replace("**", "")
            match = re.search(pattern, cleaned, re.DOTALL)
        if match:
            print(f"\n[{agent_name}]")
            print(match.group(0).strip())
        else:
            print(f"\n[{agent_name}] WARNING: Metrics block not found!")
print("=" * 60)

  EXPERIMENT 2 COMPLETE
  Model:        claude-sonnet-4-5
  Framework:    LangChain
  Interaction:  Round Robin (Next Speaker Selection)
  Total Agents: 7
  Total Steps:  7 (one per Waterfall phase)
  Total Tokens: ~37,423
  Total Time:   1476.3s

  Agent Breakdown:
    Customer_Proxy                 | Tokens: ~ 7,039 | Response: 55,370 chars
    Project_Manager                | Tokens: ~   791 | Response:  1,617 chars
    Requirement_Engineer           | Tokens: ~ 5,812 | Response: 37,304 chars
    System_Engineer                | Tokens: ~ 7,559 | Response: 67,344 chars
    Software_Developer             | Tokens: ~ 3,397 | Response: 16,338 chars
    Test_Engineer                  | Tokens: ~ 9,893 | Response: 57,364 chars
    Documentation_Engineer         | Tokens: ~ 2,932 | Response: 14,872 chars

  Output Directory: Phase1_Experiment2_Results_LangChain/

  PHASE METRICS SUMMARY

[Requirement_Engineer]
=== REQUIREMENTS PHASE METRICS ===
Total Use Cases: 18
Total Functional Require